## Stage 8 — Noise Robustness (Google Colab)

This notebook sets up everything needed to run `scripts/run_noise_robustness.py` in Colab.

### Runtime
- Colab: **Runtime → Change runtime type → GPU** (recommended)
- TPU (v5e-1) is **not supported** by this pipeline without a `torch_xla` port.

### What you need in Google Drive
Create this folder (or adjust the paths in the sync cell):

- `MyDrive/audio_fraud_detection/data/processed/real/*.wav`
- `MyDrive/audio_fraud_detection/data/processed/fake/*.wav`
- `MyDrive/audio_fraud_detection/results/split.json` (**must be the exact same split used in previous stages**)

Optional (for clean baselines in the final JSON):
- `MyDrive/audio_fraud_detection/results/transformer_results.json`
- `MyDrive/audio_fraud_detection/results/fusion_models_results.json`

### Tip
Start with a **small subset** (e.g., `white` at `20,10` dB) to validate the setup, then run full Stage 8.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone repo
%cd /content
!rm -rf /content/audio_fraud_detection
!git clone https://github.com/hadarliav1/audio_fraud_detection.git
%cd /content/audio_fraud_detection


In [ ]:
# Install dependencies
!pip -q install -r requirements.txt


In [ ]:
# Sync required files from Drive into the repo layout
# Adjust DRIVE_ROOT if your folder name is different.
from pathlib import Path
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/audio_fraud_detection')
assert DRIVE_ROOT.exists(), f"Drive folder not found: {DRIVE_ROOT}"

# Copy processed audio
src_processed = DRIVE_ROOT / 'data' / 'processed'
dst_processed = Path('/content/audio_fraud_detection/data/processed')
assert src_processed.exists(), f"Missing: {src_processed}"
if dst_processed.exists():
    shutil.rmtree(dst_processed)
shutil.copytree(src_processed, dst_processed)
print('Copied', src_processed, '->', dst_processed)

# Copy split.json (mandatory)
src_split = DRIVE_ROOT / 'results' / 'split.json'
dst_results = Path('/content/audio_fraud_detection/results')
dst_results.mkdir(parents=True, exist_ok=True)
assert src_split.exists(), f"Missing: {src_split}"
shutil.copy2(src_split, dst_results / 'split.json')
print('Copied', src_split, '->', dst_results / 'split.json')

# Optional baseline JSONs
for name in ['transformer_results.json', 'fusion_models_results.json']:
    p = DRIVE_ROOT / 'results' / name
    if p.exists():
        shutil.copy2(p, dst_results / name)
        print('Copied', p, '->', dst_results / name)

# Quick checks
print('n_processed_files =', len(list(dst_processed.rglob('*.wav'))))
print('split.json exists =', (dst_results / 'split.json').exists())


## Run Stage 8

### Option A (recommended first): Run a small subset
This validates everything works before you commit to the full run.


In [ ]:
# Write an override to limit runtime (edit as you want)
import json
from pathlib import Path

override = {
  'NOISE_TYPES': ['white'],
  'SNR_LEVELS_NOISY': [20, 10],
  'MODELS': ['hubert_base_ls960', 'wavlm'],
}
Path('results').mkdir(exist_ok=True)
Path('results/stage8_colab_override.json').write_text(json.dumps(override, indent=2))
print('Wrote results/stage8_colab_override.json')


In [ ]:
# Run Stage 8
!python3 scripts/run_noise_robustness.py


### Option B: Full Stage 8 (all noise types × SNRs)

Remove the override file, then re-run the script.

**Warning:** This can take many hours even on a GPU runtime.


In [ ]:
# Full run: delete override and run again
!rm -f results/stage8_colab_override.json
!python3 scripts/run_noise_robustness.py


In [ ]:
# View outputs
from pathlib import Path
print('noise_robustness.json exists:', Path('results/noise_robustness.json').exists())
print('comparison plot exists:', Path('reports/noise_robustness_comparison.png').exists())


## Save results back to Drive (optional)

This copies `results/noise_robustness.json` and the comparison plot to Drive.


In [ ]:
from pathlib import Path
import shutil

DRIVE_ROOT = Path('/content/drive/MyDrive/audio_fraud_detection')
out_dir = DRIVE_ROOT / 'results_stage8_colab'
out_dir.mkdir(parents=True, exist_ok=True)

for p in ['results/noise_robustness.json', 'reports/noise_robustness_comparison.png']:
    src = Path('/content/audio_fraud_detection') / p
    if src.exists():
        shutil.copy2(src, out_dir / src.name)
        print('Saved', src, '->', out_dir / src.name)
    else:
        print('Missing:', src)
